# Collaborative Filtering at Scale with PySpark ALS

This notebook ports the collaborative filtering part of the project to Apache Spark:

1. Exploratory aggregations in **Spark SQL**.
2. **Explicit ALS** (Spark MLlib) for rating prediction, scored by test RMSE on the shared 80/20 rating split.
3. **Implicit ALS** for enrolment prediction, scored by HitRate@10 and NDCG@10 on the shared leave-one-out split.
4. A comparison with the Surprise and neighbourhood models from `collaborative_filtering.ipynb`.

The splits come from `recsys/splits.py`, the same code every other notebook uses, so Spark sees exactly the same
train and test rows and the results are directly comparable. Run `collaborative_filtering.ipynb` first for section 4.

Spark runs locally here (`local[*]`), which is enough for 233k ratings. The same code runs unchanged on a cluster.

In [1]:
import os
import sys

import numpy as np
import pandas as pd
import pyspark

os.environ["SPARK_HOME"] = os.path.dirname(pyspark.__file__)
os.environ["PYSPARK_PYTHON"] = sys.executable

from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from sklearn.model_selection import train_test_split

sys.path.append("..")
from recsys import data, metrics, recommenders, results, splits

spark = (
    SparkSession.builder.master("local[*]")
    .appName("course-recommender-als")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version} running locally on {spark.sparkContext.defaultParallelism} cores")

Spark 4.2.0 running locally on 16 cores


## 1. Exploratory analysis in Spark SQL

The CSV files are read straight into Spark DataFrames and registered as SQL views.

In [2]:
data.load_ratings()
data.load_course_genres()

ratings_sdf = spark.read.csv(str(data.DATA_DIR / "ratings.csv"), header=True, schema="user INT, item STRING, rating INT")
courses_sdf = spark.read.csv(str(data.DATA_DIR / "course_genre.csv"), header=True, inferSchema=True)
courses_sdf = courses_sdf.withColumn("TITLE", F.trim(F.regexp_replace("TITLE", r"\s+", " ")))

ratings_sdf.createOrReplaceTempView("ratings")
courses_sdf.createOrReplaceTempView("courses")
print("ratings:", ratings_sdf.dtypes)
print("courses:", len(courses_sdf.columns), "columns")

ratings: [('user', 'int'), ('item', 'string'), ('rating', 'int')]
courses: 16 columns


In [3]:
spark.sql('''
    SELECT COUNT(*)                                                      AS enrolments,
           COUNT(DISTINCT user)                                          AS users,
           COUNT(DISTINCT item)                                          AS courses,
           ROUND(COUNT(*) / (COUNT(DISTINCT user) * COUNT(DISTINCT item)), 4) AS density
    FROM ratings
''').toPandas()

,enrolments,users,courses,density
0,233306,33901,126,0.0546


### Rating distribution

In [4]:
spark.sql('''
    SELECT rating,
           COUNT(*)                                     AS enrolments,
           ROUND(COUNT(*) / SUM(COUNT(*)) OVER (), 3)   AS share
    FROM ratings
    GROUP BY rating
    ORDER BY rating
''').toPandas()

,rating,enrolments,share
0,3,77866,0.334
1,4,77936,0.334
2,5,77504,0.332


### Enrolments per user

In [5]:
spark.sql('''
    WITH per_user AS (
        SELECT user, COUNT(*) AS courses FROM ratings GROUP BY user
    )
    SELECT ROUND(AVG(courses), 2)                          AS mean_courses,
           percentile_approx(courses, 0.5)                 AS median_courses,
           MAX(courses)                                    AS max_courses,
           SUM(CASE WHEN courses = 1 THEN 1 ELSE 0 END)    AS single_course_users
    FROM per_user
''').toPandas()

,mean_courses,median_courses,max_courses,single_course_users
0,6.88,6,61,8320


### Most popular courses
A window function gives each course's share of all enrolments.

In [6]:
spark.sql('''
    WITH counts AS (
        SELECT item, COUNT(*) AS enrolments FROM ratings GROUP BY item
    )
    SELECT c.item,
           k.TITLE                                               AS title,
           c.enrolments,
           ROUND(c.enrolments / SUM(c.enrolments) OVER (), 3)    AS share
    FROM counts c
    JOIN courses k ON c.item = k.COURSE_ID
    ORDER BY c.enrolments DESC
    LIMIT 10
''').toPandas()

,item,title,enrolments,share
0,PY0101EN,python for data science,14936,0.064
1,DS0101EN,introduction to data science,14477,0.062
2,BD0101EN,big data 101,13291,0.057
3,BD0111EN,hadoop 101,10599,0.045
4,DA0101EN,data analysis with python,8303,0.036
5,DS0103EN,data science methodology,7719,0.033
6,ML0101ENv3,machine learning with python,7644,0.033
7,BD0211EN,spark fundamentals i,7551,0.032
8,DS0105EN,data science hands on with open source tools,7199,0.031
9,BC0101EN,blockchain essentials,6719,0.029


### Courses per genre
The genre flags are separate columns, so `stack` unpivots them into (genre, flag) rows before aggregating.

In [7]:
genre_columns = [c for c in courses_sdf.columns if c not in ("COURSE_ID", "TITLE")]
stack_args = ", ".join(f"'{g}', {g}" for g in genre_columns)

spark.sql(f'''
    SELECT genre, SUM(flag) AS courses
    FROM courses
    LATERAL VIEW stack({len(genre_columns)}, {stack_args}) AS genre, flag
    GROUP BY genre
    ORDER BY courses DESC
''').toPandas()

,genre,courses
0,BackendDev,78
1,MachineLearning,69
2,Database,59
3,DataAnalysis,56
4,DataScience,39
5,CloudComputing,37
6,BigData,35
7,Python,28
8,FrontendDev,19
9,R,17


### Average rating per course
If some courses were rated consistently higher than others, their averages would spread out well beyond sampling noise.

In [8]:
spark.sql('''
    WITH per_course AS (
        SELECT item, AVG(rating) AS avg_rating, COUNT(*) AS n
        FROM ratings
        GROUP BY item
        HAVING COUNT(*) >= 500
    )
    SELECT COUNT(*)                        AS courses_with_500_plus,
           ROUND(MIN(avg_rating), 3)        AS min_avg,
           ROUND(MAX(avg_rating), 3)        AS max_avg,
           ROUND(STDDEV(avg_rating), 4)     AS std_of_avg,
           ROUND(SQRT(2.0 / 3 / MIN(n)), 4) AS max_expected_std_from_noise
    FROM per_course
''').toPandas()

,courses_with_500_plus,min_avg,max_avg,std_of_avg,max_expected_std_from_noise
0,71,3.925,4.045,0.0211,0.0359


Every popular course averages close to 4.0, with a spread no larger than random noise would produce
(the standard deviation of a single rating is about 0.82). This matches the permutation tests in
`model_comparison.ipynb`: the ratings do not depend on the course.

## 2. Explicit ALS: rating prediction

ALS factorises the user x course rating matrix into user and course factor vectors. MLlib needs integer ids, which come
from the shared `recsys.data.Index`. Hyperparameters are chosen on the same 10% validation slice of the training set that
the neural networks use. MLlib's ALS has no bias terms, so a prediction is just the dot product of two factor vectors,
and strong regularisation would pull it towards 0 rather than towards a typical rating. The ratings are therefore centred
on the training mean before fitting, and the mean is added back to every prediction. Users with no training ratings get
the training mean, so every test row is scored as in the other notebooks, and predictions are clipped to the 3 to 5 scale.

In [9]:
ratings = data.load_ratings()
index = data.Index(ratings)


def to_spark(df, values):
    users, items = index.encode(df)
    pdf = pd.DataFrame({"user_idx": users.astype("int32"), "item_idx": items.astype("int32"),
                        "rating": np.asarray(values, dtype="float64")})
    return spark.createDataFrame(pdf)


train, test = splits.rating_split(ratings)
fit, val = train_test_split(train, test_size=0.1, random_state=splits.SEED)
fit_mean = float(fit["rating"].mean())
train_mean = float(train["rating"].mean())
fit_sdf = to_spark(fit, fit["rating"] - fit_mean).cache()
train_sdf = to_spark(train, train["rating"] - train_mean).cache()
val_sdf = to_spark(val, val["rating"]).cache()
test_sdf = to_spark(test, test["rating"]).cache()
print(f"Fit: {fit_sdf.count():,} | validation: {val_sdf.count():,} | train: {train_sdf.count():,} | test: {test_sdf.count():,}")

Fit: 167,979 | validation: 18,665 | train: 186,644 | test: 46,662


In [10]:
evaluator = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction")


def explicit_als(rank, reg):
    return ALS(userCol="user_idx", itemCol="item_idx", ratingCol="rating", rank=rank, regParam=reg,
               maxIter=15, coldStartStrategy="nan", seed=splits.SEED)


def predict(model, sdf, mean):
    prediction = F.nanvl(F.col("prediction"), F.lit(0.0)) + F.lit(mean)
    clipped = F.least(F.greatest(prediction, F.lit(float(data.RATING_MIN))), F.lit(float(data.RATING_MAX)))
    return model.transform(sdf).withColumn("prediction", clipped)


grid = []
for rank in (5, 10, 20):
    for reg in (0.05, 0.1, 0.5, 1.0, 5.0):
        model = explicit_als(rank, reg).fit(fit_sdf)
        grid.append({"rank": rank, "regParam": reg, "val_rmse": evaluator.evaluate(predict(model, val_sdf, fit_mean))})

grid = pd.DataFrame(grid).sort_values("val_rmse")
best = grid.iloc[0]
print(f"Selected rank {int(best['rank'])}, regParam {best['regParam']}")
grid.head(6).round(4)

Selected rank 5, regParam 0.5


,rank,regParam,val_rmse
2,5,0.5,0.8187
7,10,0.5,0.8187
12,20,0.5,0.8187
8,10,1.0,0.8187
3,5,1.0,0.8187
4,5,5.0,0.8187


In [11]:
als_model = explicit_als(int(best["rank"]), float(best["regParam"])).fit(train_sdf)
test_predictions = predict(als_model, test_sdf, train_mean)

als_rmse = evaluator.evaluate(test_predictions)
check = test_predictions.select("rating", "prediction").toPandas()
print(f"Explicit ALS test RMSE: {als_rmse:.4f}")
print(f"Same RMSE with the shared numpy metric: {metrics.rmse(check['rating'], check['prediction']):.4f}")
print(f"Global mean baseline test RMSE: {metrics.rmse(test['rating'], train_mean):.4f}")

Explicit ALS test RMSE: 0.8124
Same RMSE with the shared numpy metric: 0.8124
Global mean baseline test RMSE: 0.8123


Like every other rating model in the project, ALS cannot beat the global mean. Weak regularisation fits noise and does
worse. With enough regularisation the factors shrink to zero and ALS predicts the training mean, so it ties with the baseline.

## 3. Implicit ALS: enrolment prediction

Implicit ALS treats each enrolment as a positive signal of preference (value 1) and every missing enrolment as a weak
negative, weighted by `alpha`. The ratings are not used, since they carry no signal. After fitting, the user and course
factors are collected and every course is scored for every test user as the dot product of their factor vectors.
Hyperparameters are chosen on the validation enrolments, never on the test set.

In [12]:
train_loo, test_loo = splits.leave_one_out(ratings)
fit_loo, val_loo = splits.leave_one_out(train_loo, seed=splits.SEED + 1)


def factor_matrix(factors_sdf, n_rows):
    pdf = factors_sdf.toPandas()
    matrix = np.zeros((n_rows, len(pdf["features"].iloc[0])))
    matrix[pdf["id"].to_numpy()] = np.vstack(pdf["features"].to_numpy())
    return matrix


def implicit_als_metrics(fit_df, eval_df, rank, alpha, reg=0.1):
    enrolments = to_spark(fit_df, np.ones(len(fit_df)))
    model = ALS(userCol="user_idx", itemCol="item_idx", ratingCol="rating", implicitPrefs=True,
                rank=rank, alpha=alpha, regParam=reg, maxIter=15, seed=splits.SEED).fit(enrolments)
    user_factors = factor_matrix(model.userFactors, index.n_users)
    item_factors = factor_matrix(model.itemFactors, index.n_items)
    users, targets = index.encode(eval_df)
    seen = splits.interaction_matrix(fit_df, index)[users].toarray() > 0
    return metrics.ranking_metrics(user_factors[users] @ item_factors.T, seen, targets)


grid = []
for rank in (8, 16, 32, 64):
    for reg in (0.1, 1.0, 3.0):
        for alpha in (1.0, 10.0, 40.0):
            scores = implicit_als_metrics(fit_loo, val_loo, rank, alpha, reg)
            grid.append({"rank": rank, "regParam": reg, "alpha": alpha,
                         "val_ndcg@10": scores["ndcg@10"], "val_hit_rate@10": scores["hit_rate@10"]})

grid = pd.DataFrame(grid).sort_values("val_ndcg@10", ascending=False)
best_implicit = grid.iloc[0]
print(f"Selected rank {int(best_implicit['rank'])}, regParam {best_implicit['regParam']}, alpha {best_implicit['alpha']}")
grid.head(8).round(4)

Selected rank 32, regParam 1.0, alpha 10.0


,rank,regParam,alpha,val_ndcg@10,val_hit_rate@10
22,32,1.0,10.0,0.4794,0.7380
31,64,1.0,10.0,0.4788,0.7479
35,64,3.0,40.0,0.4765,0.7420
13,16,1.0,10.0,0.4741,0.7386
26,32,3.0,40.0,0.4728,0.7407
32,64,1.0,40.0,0.4712,0.7453
17,16,3.0,40.0,0.4596,0.7329
4,8,1.0,10.0,0.4543,0.7286


In [13]:
implicit_scores = implicit_als_metrics(train_loo, test_loo, int(best_implicit["rank"]), float(best_implicit["alpha"]),
                                      float(best_implicit["regParam"]))
print(f"Implicit ALS test HitRate@10: {implicit_scores['hit_rate@10']:.4f} | NDCG@10: {implicit_scores['ndcg@10']:.4f}")

Implicit ALS test HitRate@10: 0.7734 | NDCG@10: 0.5480


## 4. Comparison with the Surprise and neighbourhood models

The ALS results are added to the shared results files, then compared with the collaborative filtering models from
`collaborative_filtering.ipynb` and with the simple baselines, all on the same test sets.

In [14]:
als_label = f"ALS explicit, PySpark (rank {int(best['rank'])}, reg {best['regParam']})"
implicit_label = f"ALS implicit, PySpark (rank {int(best_implicit['rank'])}, reg {best_implicit['regParam']}, alpha {best_implicit['alpha']})"
results.save([{"family": "Collaborative filtering", "model": als_label, "test_rmse": als_rmse}], "rating_rmse", source="als_pyspark")
results.save([{"family": "Collaborative filtering", "model": implicit_label, **implicit_scores}], "ranking", source="als_pyspark")

rating = results.load("rating_rmse")
rating = pd.concat([
    rating[rating["family"] == "Collaborative filtering"],
    pd.DataFrame([{"family": "Baseline", "model": "Global mean", "test_rmse": metrics.rmse(test["rating"], train_mean)}]),
])
rating.sort_values("test_rmse").round(4)

,family,model,test_rmse,source
0,Baseline,Global mean,0.8123,NaN
26,Collaborative filtering,"ALS explicit, PySpark (rank 5, reg 0.5)",0.8124,als_pyspark
21,Collaborative filtering,Baseline (user + item bias),0.8181,collaborative_filtering
22,Collaborative filtering,KNN user-based,0.8222,collaborative_filtering
25,Collaborative filtering,SVD,0.8253,collaborative_filtering
23,Collaborative filtering,KNN item-based,0.8786,collaborative_filtering
24,Collaborative filtering,NMF,0.9391,collaborative_filtering


In [15]:
train_matrix = splits.interaction_matrix(train_loo, index)
users, targets = index.encode(test_loo)
seen = train_matrix[users].toarray() > 0
popularity = metrics.ranking_metrics(recommenders.popularity_scores(train_matrix, len(users)), seen, targets)

ranking = results.load("ranking")
ranking = pd.concat([
    ranking[ranking["family"] == "Collaborative filtering"],
    pd.DataFrame([{"family": "Baseline", "model": "Most popular", **popularity}]),
])
ranking.sort_values("ndcg@10", ascending=False).round(4)

,family,model,hit_rate@10,ndcg@10,source
15,Collaborative filtering,User KNN (k=200),0.7856,0.5630,collaborative_filtering
18,Collaborative filtering,"ALS implicit, PySpark (rank 32, reg 1.0, alpha...",0.7734,0.5480,als_pyspark
14,Collaborative filtering,Item KNN (k=125),0.7379,0.5248,collaborative_filtering
16,Collaborative filtering,PureSVD (rank=4),0.6946,0.4712,collaborative_filtering
0,Baseline,Most popular,0.5728,0.3408,NaN


Rating prediction: once regularised enough, explicit ALS ties with the global mean baseline, and none of the Surprise
models does better than the baseline either. A second library confirms that the ratings carry no learnable signal.

Enrolment prediction: implicit ALS is compared with the Surprise-based and neighbourhood models above. Unlike ranking
by a predicted rating, it learns from which courses users take, so it is a genuine recommender.

In [16]:
spark.stop()